# N130 · 线性递推与矩阵快速幂

**目标：** 把固定维状态递推写成线性变换。

**先修：** N098, N128, N082。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 状态向量；转移矩阵；矩阵乘法；单位矩阵；模数；维度成本。

## 从问题到算法

若状态满足线性递推x[t+1]=A·x[t]，那么多步转移就是A的幂。二进制快速幂适用于矩阵，因为矩阵乘法满足结合律；一般矩阵不交换，所以不能随意改变乘法次序。状态维度必须固定且小，否则矩阵乘法本身可能比线性DP更贵。

## 最小实现

**本章接口：** `matmul_mod(a, b, mod)`、`matpow_mod(a, n, mod)`、`fib_matrix(n)`、`count_vowel_permutation(n)`

In [1]:
def matmul_mod(a,b,mod):
    if mod is not None and mod<=0: raise ValueError('positive modulus required')
    if not a or not b: return []
    rows,inner,cols=len(a),len(a[0]),len(b[0])
    if any(len(row)!=inner for row in a) or len(b)!=inner or any(len(row)!=cols for row in b): raise ValueError('incompatible rectangular matrices')
    out=[[0]*cols for _ in range(rows)]
    for i in range(rows):
        for k in range(inner):
            for j in range(cols):
                out[i][j]+=a[i][k]*b[k][j]
                if mod is not None: out[i][j]%=mod
    return out

def matpow_mod(a,n,mod):
    size=len(a)
    if n<0 or any(len(row)!=size for row in a): raise ValueError('square matrix and nonnegative exponent required')
    result=[[int(i==j) if mod is None else int(i==j)%mod for j in range(size)] for i in range(size)]; base=[row[:] for row in a]
    while n:
        if n&1: result=matmul_mod(result,base,mod)
        base=matmul_mod(base,base,mod); n>>=1
    return result

def fib_matrix(n):
    if n<0: raise ValueError('nonnegative index required')
    return matpow_mod([[1,1],[1,0]],n,None)[0][1]

def count_vowel_permutation(n):
    if n<0: raise ValueError('nonnegative length required')
    if n==0: return 1
    edges={0:[1],1:[0,2],2:[0,1,3,4],3:[2,4],4:[0]}; matrix=[[0]*5 for _ in range(5)]
    for old,nexts in edges.items():
        for new in nexts: matrix[new][old]=1
    power=matpow_mod(matrix,n-1,10**9+7)
    return sum(map(sum,power))%(10**9+7)

## 正确性、前提与复杂度

转移矩阵的列代表旧状态、行代表新状态。乘法展开正好累加所有一步来源，矩阵幂相乘拼接独立转移步。快速幂保持已累积结果与尚未处理的幂因子乘积不变；零次幂必须为单位阵而非零阵。

**代价：** d维稠密矩阵幂O(d³ log n)次算术操作、O(d²)空间。fib_matrix用mod=None得到精确整数，结果位数为Θ(n)，因此不能称总位复杂度O(log n)。元音计数采用模10^9+7。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

matrix=[[1,1],[1,0]]
show_table(['指数','矩阵幂','F(n)'],[(n,matpow_mod(matrix,n,None),fib_matrix(n)) for n in range(6)])

指数,矩阵幂,F(n)
0,"[[1, 0], [0, 1]]",0
1,"[[1, 1], [1, 0]]",1
2,"[[2, 1], [1, 1]]",1
3,"[[3, 2], [2, 1]]",2
4,"[[5, 3], [3, 2]]",3
5,"[[8, 5], [5, 3]]",5


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert matpow_mod([[2,3],[4,5]],0,17)==[[1,0],[0,1]]
assert fib_matrix(0)==0 and fib_matrix(10)==55
assert count_vowel_permutation(1)==5 and count_vowel_permutation(2)==10 and count_vowel_permutation(5)==68
x,y=0,1
for n in range(120): assert fib_matrix(n)==x; x,y=y,x+y
edges={0:[1],1:[0,2],2:[0,1,3,4],3:[2,4],4:[0]}; dp=[1]*5
for n in range(1,40):
    assert count_vowel_permutation(n)==sum(dp)%(10**9+7)
    nxt=[0]*5
    for old,targets in edges.items():
        for new in targets: nxt[new]+=dp[old]
    dp=nxt
assert matmul_mod([[1,2,3]],[[4],[5],[6]],97)==[[32]]
print('N130: 所有本章断言通过')

N130: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 从三阶递推构造矩阵。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明复杂度含矩阵维度而不只log n。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 509. Fibonacci Number（extension）
- 1137. N-th Tribonacci Number（extension）
- 1220. Count Vowels Permutation（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。